In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random
import os
import math

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ======================================================
# [V12] Y축 데이터 증강 + MDN
# ======================================================
BATCH_SIZE = 64
LR_BASE = 1e-3
WARMUP_EPOCHS = 3
EPOCHS_BASE = 50
DROPOUT = 0.2
MAX_SEQ_LEN = 30
GRAD_CLIP = 1.0

HIDDEN_DIM = 256
LSTM_LAYERS = 3
BIDIRECTIONAL = True

# MDN 파라미터
NUM_GAUSSIANS = 10
MIN_SIGMA = 0.01
MAX_SIGMA = 0.3
HYBRID_LOSS_WEIGHT = 0.3

print(f"[V12] Y축 데이터 증강 + MDN")
print(f"  주요 기능:")
print(f"  1. ✅ Y축 반전 데이터 증강 (사이드라인 대칭성 활용)")
print(f"  2. MDN (2 Gaussians)")
print(f"  3. Spatial-Temporal Attention")
print(f"  4. Hybrid Loss (NLL + MSE)")

Device: cuda
[V12] Y축 데이터 증강 + MDN
  주요 기능:
  1. ✅ Y축 반전 데이터 증강 (사이드라인 대칭성 활용)
  2. MDN (2 Gaussians)
  3. Spatial-Temporal Attention
  4. Hybrid Loss (NLL + MSE)


c:\Anaconda3\envs\daycon\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ======================================================
# 데이터 로드 (증강은 나중에!)
# ======================================================
BASE_DIR = "./open_track1"
if not os.path.exists(BASE_DIR): BASE_DIR = "."

TRAIN_PATH = os.path.join(BASE_DIR, "train.csv")
TEST_META_PATH = os.path.join(BASE_DIR, "test.csv")
MATCH_PATH = os.path.join(BASE_DIR, "match_info.csv")

train_df = pd.read_csv(TRAIN_PATH)
print(f"✅ Train Loaded: {train_df.shape}")

if os.path.exists(TEST_META_PATH):
    test_meta = pd.read_csv(TEST_META_PATH)
    print(f"ℹ️ Reading {len(test_meta)} test files...")
    test_dfs = []
    for _, row in tqdm(test_meta.iterrows(), total=len(test_meta), desc="Loading Test CSVs"):
        rel_path = row['path']
        paths_to_try = [
            rel_path,
            os.path.join(BASE_DIR, rel_path.lstrip("./")),
            os.path.join(BASE_DIR, "test", str(row['game_id']), os.path.basename(rel_path))
        ]
        for p in paths_to_try:
            if os.path.exists(p):
                test_dfs.append(pd.read_csv(p))
                break
    if test_dfs:
        test_df = pd.concat(test_dfs, ignore_index=True)
        print(f"✅ Test Data Merged: {test_df.shape}")
else:
    raise FileNotFoundError("test.csv not found")

if os.path.exists(MATCH_PATH):
    match_info = pd.read_csv(MATCH_PATH)
    match_subset = match_info[['game_id', 'home_team_id', 'venue']]
    train_df = pd.merge(train_df, match_subset, on='game_id', how='left')
    test_df = pd.merge(test_df, match_subset, on='game_id', how='left')

def preprocess(df):
    if 'home_team_id' in df.columns:
        df['is_home'] = (df['team_id'] == df['home_team_id']).astype(float)
    else:
        df['is_home'] = 0.5
    if 'end_x' not in df.columns:
        df['end_x'] = 0.0
        df['end_y'] = 0.0
    else:
        df['end_x'] = df['end_x'].fillna(0.0)
        df['end_y'] = df['end_y'].fillna(0.0)
    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

ID_COL = 'game_episode' if 'game_episode' in train_df.columns else 'episode_id'
print(f"\nData Ready. ID Column: {ID_COL}")
print(f"⚠️ 데이터 증강은 train/val split 후에 적용됩니다 (데이터 유출 방지)")

✅ Train Loaded: (356721, 15)
ℹ️ Reading 2414 test files...


Loading Test CSVs: 100%|██████████| 2414/2414 [00:04<00:00, 553.90it/s]


✅ Test Data Merged: (53110, 15)

Data Ready. ID Column: game_episode
⚠️ 데이터 증강은 train/val split 후에 적용됩니다 (데이터 유출 방지)


In [4]:
# ======================================================
# Team ID 범위 확인 및 매핑
# ======================================================
print("\n" + "="*70)
print("🔍 Team ID 분석")
print("="*70)

# 실제 team_id 값 확인
unique_teams_train = sorted(train_df['team_id'].unique())
unique_teams_test = sorted(test_df['team_id'].unique())
all_unique_teams = sorted(set(list(unique_teams_train) + list(unique_teams_test)))

print(f"Train unique team_ids: {unique_teams_train}")
print(f"Test unique team_ids: {unique_teams_test}")
print(f"All unique team_ids: {all_unique_teams}")
print(f"Min: {min(all_unique_teams)}, Max: {max(all_unique_teams)}")
print(f"Total unique teams: {len(all_unique_teams)}")

# Team ID를 0부터 시작하는 연속된 정수로 매핑
TEAM_ID_MAPPING = {tid: idx + 1 for idx, tid in enumerate(all_unique_teams)}  # 1부터 시작 (0은 padding)
TEAM_ID_MAPPING[0] = 0  # padding

print(f"\n✅ Team ID Mapping (원본 → 인덱스):")
for orig, mapped in sorted(TEAM_ID_MAPPING.items()):
    if orig != 0:
        print(f"   {orig} → {mapped}")

# 데이터프레임에 매핑 적용
train_df['team_id_mapped'] = train_df['team_id'].map(TEAM_ID_MAPPING)
test_df['team_id_mapped'] = test_df['team_id'].map(TEAM_ID_MAPPING)

# 매핑 후 확인
print(f"\n✅ 매핑 후 범위: 0 (padding) ~ {len(all_unique_teams)}")
print(f"   Embedding vocab size: {len(all_unique_teams) + 1}")
print("="*70)

# 전역 변수로 저장
NUM_TEAMS_ACTUAL = len(all_unique_teams)


🔍 Team ID 분석
Train unique team_ids: [316, 2353, 2354, 4220, 4639, 4640, 4641, 4643, 4644, 4646, 4648, 4657]
Test unique team_ids: [316, 2353, 2354, 4220, 4639, 4640, 4641, 4643, 4644, 4646, 4648, 4657]
All unique team_ids: [316, 2353, 2354, 4220, 4639, 4640, 4641, 4643, 4644, 4646, 4648, 4657]
Min: 316, Max: 4657
Total unique teams: 12

✅ Team ID Mapping (원본 → 인덱스):
   316 → 1
   2353 → 2
   2354 → 3
   4220 → 4
   4639 → 5
   4640 → 6
   4641 → 7
   4643 → 8
   4644 → 9
   4646 → 10
   4648 → 11
   4657 → 12

✅ 매핑 후 범위: 0 (padding) ~ 12
   Embedding vocab size: 13


In [5]:
# ======================================================
# [V15] 피처 엔지니어링 + N-gram (type + result 조합)
# ======================================================
TOP_TYPES = ['Pass', 'Carry', 'Recovery', 'Interception', 'Duel', 'Tackle', 
             'Throw-In', 'Clearance', 'Intervention', 'Block', 'Pass_Freekick', 
             'Cross', 'Goal Kick', 'Error', 'Shot']
ALL_RESULTS = ['Successful', 'Unsuccessful', 'On Target', 'Yellow_Card', 
               'Blocked', 'Keeper Rush-Out', 'Low Quality Shot', 'Off Target']

# 🆕 N-gram 패턴 (type + result 조합)
TOP_3GRAMS = []
TOP_5GRAMS = []
NGRAM_3_SIZE = 20
NGRAM_5_SIZE = 20

# 패턴 to index 매핑 (Embedding용)
PATTERN_3_TO_IDX = {}
PATTERN_5_TO_IDX = {}

def extract_ngrams_from_data(df, n_size=3, top_k=20):
    """
    type_name + result_name 조합으로 N-gram 패턴 추출
    
    예: "Pass_Successful" -> "Carry_Successful" -> "Pass_Successful"
    """
    from collections import Counter
    patterns = []

    for _, group in df.groupby(ID_COL, sort=False):
        # type + result 조합
        combined = [
            f"{t}_{r}" if pd.notna(r) and r else t
            for t, r in zip(group['type_name'].values, group['result_name'].values)
        ]
        
        for i in range(n_size - 1, len(combined)):
            pattern = tuple(combined[i - n_size + 1 : i + 1])
            patterns.append(pattern)

    # 빈도 계산 및 상위 K개 추출
    counter = Counter(patterns)
    top_patterns = [p for p, _ in counter.most_common(top_k)]
    
    print(f"\n🔍 {n_size}-gram 패턴 분석 (type+result):")
    print(f"   전체 유니크 패턴: {len(counter)}")
    print(f"   상위 {top_k}개:")
    for i, (pattern, count) in enumerate(counter.most_common(min(10, top_k)), 1):
        pattern_str = ' → '.join(pattern)
        print(f"      {i}. {pattern_str}: {count:,}회")
    
    return top_patterns


def make_features(group):
    """
    피처 생성 + N-gram 인덱스 반환
    """
    n = len(group)
    sx = group['start_x'].values / 105.0
    sy = group['start_y'].values / 68.0
    ex = group['end_x'].values / 105.0
    ey = group['end_y'].values / 68.0
    is_home = group['is_home'].values
    
    if 'time_seconds' in group.columns:
        times = group['time_seconds'].values
        dt = np.zeros(n, dtype=np.float32)
        dt[1:] = times[1:] - times[:-1]
        dt = np.maximum(dt, 0.1)
    else:
        dt = np.ones(n, dtype=np.float32)

    dx = ex - sx
    dy = ey - sy
    dist_meter = np.sqrt((dx*105)**2 + (dy*68)**2)
    cumsum_dx = np.cumsum(dx) / 105.0
    cumsum_dy = np.cumsum(dy) / 68.0
    lag_dist_m = np.roll(dist_meter, 1); lag_dist_m[0] = 0
    lag_cumsum_dx = np.roll(cumsum_dx, 1); lag_cumsum_dx[0] = 0
    lag_cumsum_dy = np.roll(cumsum_dy, 1); lag_cumsum_dy[0] = 0
    lag_dt = np.roll(dt, 1); lag_dt[0] = 1.0
    lag_speed = lag_dist_m / np.maximum(lag_dt, 0.1)
    
    if 'player_id' in group.columns:
        p_ids = group['player_id'].values
        is_same = np.zeros(n, dtype=np.float32)
        is_same[1:] = (p_ids[1:] == p_ids[:-1]).astype(np.float32)
    else:
        is_same = np.zeros(n, dtype=np.float32)

    progress = np.arange(n) / max(n-1, 1)
    is_second_half = (group['period_id'].values > 1).astype(np.float32) if 'period_id' in group.columns else np.zeros(n)
    
    GOAL_X, GOAL_Y = 105.0, 34.0
    sx_real, sy_real = sx * 105.0, sy * 68.0
    dist_to_goal = np.sqrt((sx_real - GOAL_X)**2 + (sy_real - GOAL_Y)**2) / 105.0
    angle_to_goal = np.arctan2(GOAL_Y - sy_real, GOAL_X - sx_real)
    angle_sin, angle_cos = np.sin(angle_to_goal), np.cos(angle_to_goal)
    dist_to_sideline = np.minimum(sy_real, 68.0 - sy_real) / 68.0
    dist_to_endline = np.minimum(sx_real, 105.0 - sx_real) / 105.0
    
    def get_zone(x_norm):
        if x_norm < 35.0/105.0: return 0
        elif x_norm < 70.0/105.0: return 1
        else: return 2
    
    zones = np.array([get_zone(x) for x in sx])
    zone_onehot = np.zeros((n, 3), dtype=np.float32)
    for i, z in enumerate(zones): zone_onehot[i, z] = 1.0
    
    types_onehot = np.zeros((n, len(TOP_TYPES) + 1), dtype=np.float32)
    for i, t in enumerate(group['type_name'].values):
        types_onehot[i, TOP_TYPES.index(t) if t in TOP_TYPES else -1] = 1.0
    
    results_onehot = np.zeros((n, len(ALL_RESULTS) + 1), dtype=np.float32)
    for i, r in enumerate(group['result_name'].values):
        results_onehot[i, ALL_RESULTS.index(r) if r in ALL_RESULTS else -1] = 1.0

    # 🆕 N-gram 인덱스 생성 (Embedding용)
    combined_list = [
        f"{t}_{r}" if pd.notna(r) and r else t
        for t, r in zip(group['type_name'].values, group['result_name'].values)
    ]
    
    ngram3_idx = np.zeros(n, dtype=np.int64)  # 0 = padding
    ngram5_idx = np.zeros(n, dtype=np.int64)
    
    for i in range(n):
        # 3-gram
        if i >= 2:
            pattern = tuple(combined_list[i-2:i+1])
            if pattern in PATTERN_3_TO_IDX:
                ngram3_idx[i] = PATTERN_3_TO_IDX[pattern]
            else:
                ngram3_idx[i] = len(TOP_3GRAMS) + 1  # 'Others' 인덱스
        
        # 5-gram
        if i >= 4:
            pattern = tuple(combined_list[i-4:i+1])
            if pattern in PATTERN_5_TO_IDX:
                ngram5_idx[i] = PATTERN_5_TO_IDX[pattern]
            else:
                ngram5_idx[i] = len(TOP_5GRAMS) + 1  # 'Others' 인덱스

    # 기본 피처 + N-gram 인덱스
    features = []
    ngram3_indices = []
    ngram5_indices = []
    
    for i in range(n):
        scalars = [sx[i], sy[i], lag_cumsum_dx[i], lag_cumsum_dy[i], lag_dist_m[i]/100.0,
                   lag_speed[i]/10.0, dt[i]/10.0, progress[i], is_home[i], is_same[i],
                   is_second_half[i], dist_to_goal[i], angle_sin[i], angle_cos[i],
                   dist_to_sideline[i], dist_to_endline[i]]
        feat_vec = np.concatenate([scalars, zone_onehot[i], types_onehot[i], results_onehot[i]])
        features.append(feat_vec)
        ngram3_indices.append(ngram3_idx[i])
        ngram5_indices.append(ngram5_idx[i])
        
        if i < n - 1:
            ex_real, ey_real = ex[i] * 105.0, ey[i] * 68.0
            end_dist_to_goal = np.sqrt((ex_real - GOAL_X)**2 + (ey_real - GOAL_Y)**2) / 105.0
            end_angle = np.arctan2(GOAL_Y - ey_real, GOAL_X - ex_real)
            scalars_end = scalars.copy()
            scalars_end[0:2] = [ex[i], ey[i]]
            scalars_end[2:4] = [cumsum_dx[i], cumsum_dy[i]]
            scalars_end[11:16] = [end_dist_to_goal, np.sin(end_angle), np.cos(end_angle),
                                   min(ey_real, 68.0 - ey_real) / 68.0,
                                   min(ex_real, 105.0 - ex_real) / 105.0]
            end_zone_onehot = np.zeros(3, dtype=np.float32)
            end_zone_onehot[get_zone(ex[i])] = 1.0
            feat_vec_end = np.concatenate([scalars_end, end_zone_onehot, types_onehot[i], results_onehot[i]])
            features.append(feat_vec_end)
            ngram3_indices.append(ngram3_idx[i])
            ngram5_indices.append(ngram5_idx[i])
            
    return (np.array(features, dtype=np.float32), 
            np.array(ngram3_indices, dtype=np.int64),
            np.array(ngram5_indices, dtype=np.int64))


# 🆕 N-gram 패턴 추출
print("\n" + "="*70)
print("🔍 N-gram 패턴 추출 중 (type+result 조합)...")
print("="*70)

TOP_3GRAMS = extract_ngrams_from_data(train_df, n_size=3, top_k=NGRAM_3_SIZE)
TOP_5GRAMS = extract_ngrams_from_data(train_df, n_size=5, top_k=NGRAM_5_SIZE)

# 인덱스 매핑 생성 (1부터 시작, 0은 padding)
PATTERN_3_TO_IDX = {pattern: idx + 1 for idx, pattern in enumerate(TOP_3GRAMS)}
PATTERN_5_TO_IDX = {pattern: idx + 1 for idx, pattern in enumerate(TOP_5GRAMS)}

print(f"\n✅ N-gram 패턴 추출 완료:")
print(f"   3-gram: {len(TOP_3GRAMS)}개 (+ Others)")
print(f"   5-gram: {len(TOP_5GRAMS)}개 (+ Others)")
print(f"   Embedding vocab size: 3-gram={len(TOP_3GRAMS)+2}, 5-gram={len(TOP_5GRAMS)+2}")
print(f"   (0=padding, 1~{len(TOP_3GRAMS)}=top patterns, {len(TOP_3GRAMS)+1}=others)")
print("="*70)

# INPUT_DIM 계산
dummy_group = train_df.iloc[:5].copy()
dummy_feats, _, _ = make_features(dummy_group)
INPUT_DIM = dummy_feats.shape[1]
print(f"\n✅ Base Input Dimension: {INPUT_DIM}")
print(f"🆕 N-gram은 별도 Embedding Layer로 처리 (모델에서 concat)")


🔍 N-gram 패턴 추출 중 (type+result 조합)...

🔍 3-gram 패턴 분석 (type+result):
   전체 유니크 패턴: 1584
   상위 20개:
      1. Pass_Successful → Carry → Pass_Successful: 50,130회
      2. Pass_Successful → Pass_Successful → Pass_Successful: 30,028회
      3. Carry → Pass_Successful → Carry: 27,832회
      4. Pass_Successful → Pass_Successful → Carry: 26,015회
      5. Carry → Pass_Successful → Pass_Successful: 25,251회
      6. Recovery → Carry → Pass_Successful: 6,285회
      7. Pass_Successful → Carry → Pass_Unsuccessful: 6,163회
      8. Recovery → Pass_Successful → Pass_Successful: 5,140회
      9. Recovery → Pass_Successful → Carry: 4,248회
      10. Interception → Clearance → Recovery: 3,942회

🔍 5-gram 패턴 분석 (type+result):
   전체 유니크 패턴: 12084
   상위 20개:
      1. Pass_Successful → Carry → Pass_Successful → Carry → Pass_Successful: 18,246회
      2. Carry → Pass_Successful → Carry → Pass_Successful → Carry: 10,077회
      3. Pass_Successful → Pass_Successful → Pass_Successful → Carry → Pass_Successful: 9,696회
 

In [6]:
# ======================================================
# 데이터셋 (N-gram 인덱스 + Team ID 포함)
# ======================================================

# Team ID 매핑 자동 생성 (없을 경우)
if 'team_id_mapped' not in train_df.columns:
    print("\n⚠️ team_id_mapped가 없어서 자동 생성합니다...")
    unique_teams = sorted(set(list(train_df['team_id'].unique()) + list(test_df['team_id'].unique())))
    TEAM_ID_MAPPING = {tid: idx + 1 for idx, tid in enumerate(unique_teams)}
    TEAM_ID_MAPPING[0] = 0
    train_df['team_id_mapped'] = train_df['team_id'].map(TEAM_ID_MAPPING).fillna(0).astype(int)
    test_df['team_id_mapped'] = test_df['team_id'].map(TEAM_ID_MAPPING).fillna(0).astype(int)
    NUM_TEAMS_ACTUAL = len(unique_teams)
    print(f"✅ Team ID 매핑 완료: {len(unique_teams)} teams, vocab size = {NUM_TEAMS_ACTUAL + 1}")


class SoccerDataset(Dataset):
    def __init__(self, df, mode='train', augment_y=False):
        """
        augment_y: Y축 반전 증강 여부
        """
        self.mode = mode
        self.augment_y = augment_y
        self.episodes = []
        self.ngram3_indices = []
        self.ngram5_indices = []
        self.team_ids = []  # 🆕 에피소드별 team_id (매핑된 값)
        self.targets = []
        self.episode_ids = []
        
        for name, group in tqdm(df.groupby(ID_COL, sort=False), desc=f"Dataset ({mode})"):
            if mode == 'train' and len(group) < 2: continue
            
            # 원본 추가
            seq, ng3_idx, ng5_idx = make_features(group)
            team_id = int(group.iloc[0]['team_id_mapped'])  # 🔧 매핑된 team_id 사용
            
            if mode == 'train' or mode == 'val':
                last = group.iloc[-1]
                self.targets.append([last['end_x']/105.0, last['end_y']/68.0])
                self.episodes.append(seq)
                self.ngram3_indices.append(ng3_idx)
                self.ngram5_indices.append(ng5_idx)
                self.team_ids.append(team_id)
                self.episode_ids.append(str(name))
            else:
                self.episodes.append(seq)
                self.ngram3_indices.append(ng3_idx)
                self.ngram5_indices.append(ng5_idx)
                self.team_ids.append(team_id)
                self.episode_ids.append(str(name))
            
            # 🆕 Y축 증강 (train만!)
            if mode == 'train' and augment_y:
                group_aug = group.copy()
                group_aug['start_y'] = 68.0 - group_aug['start_y']
                group_aug['end_y'] = 68.0 - group_aug['end_y']
                
                seq_aug, ng3_idx_aug, ng5_idx_aug = make_features(group_aug)
                last_aug = group_aug.iloc[-1]
                self.targets.append([last_aug['end_x']/105.0, last_aug['end_y']/68.0])
                self.episodes.append(seq_aug)
                self.ngram3_indices.append(ng3_idx_aug)
                self.ngram5_indices.append(ng5_idx_aug)
                self.team_ids.append(team_id)
                self.episode_ids.append(str(name))  # 같은 ID (증강 버전)

    def __len__(self): return len(self.episodes)
    
    def __getitem__(self, idx):
        seq = torch.FloatTensor(self.episodes[idx])
        ng3 = torch.LongTensor(self.ngram3_indices[idx])
        ng5 = torch.LongTensor(self.ngram5_indices[idx])
        team_id = self.team_ids[idx]
        episode_id = self.episode_ids[idx]
        
        if len(seq) > MAX_SEQ_LEN:
            seq = seq[-MAX_SEQ_LEN:]
            ng3 = ng3[-MAX_SEQ_LEN:]
            ng5 = ng5[-MAX_SEQ_LEN:]
        
        if self.mode == 'train' or self.mode == 'val':
            return seq, ng3, ng5, team_id, torch.FloatTensor(self.targets[idx]), episode_id
        return seq, ng3, ng5, team_id, episode_id  # test mode


def collate_fn(batch):
    """
    Collate function for N-gram + Team ID
    """
    seqs = [b[0] for b in batch]
    ng3s = [b[1] for b in batch]
    ng5s = [b[2] for b in batch]
    team_ids = [b[3] for b in batch]
    
    lens = torch.LongTensor([len(s) for s in seqs])
    
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    ng3_padded = pad_sequence(ng3s, batch_first=True, padding_value=0)
    ng5_padded = pad_sequence(ng5s, batch_first=True, padding_value=0)
    team_ids_tensor = torch.LongTensor(team_ids)
    
    mask = torch.arange(padded.size(1))[None, :] >= lens[:, None]
    
    if len(batch[0]) == 6:  # Train/Val: (seq, ng3, ng5, team_id, target, episode_id)
        targets = torch.stack([b[4] for b in batch])
        episode_ids = [b[5] for b in batch]
        return (padded, ng3_padded, ng5_padded, team_ids_tensor, targets, mask, lens, episode_ids)
    else:  # Test: (seq, ng3, ng5, team_id, episode_id) - 5개
        episode_ids = [b[4] for b in batch]
        return (padded, ng3_padded, ng5_padded, team_ids_tensor, mask, lens, episode_ids)


# 데이터셋 생성
full_dataset = SoccerDataset(train_df, mode='train', augment_y=False)
test_dataset = SoccerDataset(test_df, mode='test', augment_y=False)
print(f"✅ Dataset: {len(full_dataset)} episodes (증강 전)")
print(f"✅ Test: {len(test_dataset)} episodes")
print(f"🆕 N-gram + Team ID 포함 (매핑된 team_id 사용)")


Dataset (test): 100%|██████████| 2414/2414 [00:02<00:00, 940.39it/s]

✅ Dataset: 15428 episodes (증강 전)
✅ Test: 2414 episodes
🆕 N-gram + Team ID 포함 (매핑된 team_id 사용)


In [7]:
# ======================================================
# [V16] 물리 엔진 + 전술 직관 분리 모델
# ======================================================
from torch.optim.lr_scheduler import CosineAnnealingLR

# NUM_TEAMS_ACTUAL이 정의되지 않은 경우 자동 계산
if 'NUM_TEAMS_ACTUAL' not in globals():
    if 'team_id_mapped' in train_df.columns:
        NUM_TEAMS_ACTUAL = train_df['team_id_mapped'].max()
    else:
        NUM_TEAMS_ACTUAL = 12  # 기본값
    print(f"⚠️ NUM_TEAMS_ACTUAL 자동 설정: {NUM_TEAMS_ACTUAL}")

# Embedding 설정
NGRAM_EMBED_DIM = 6  # 3-gram, 5-gram 임베딩 차원
TEAM_EMBED_DIM = 4   # Team ID 임베딩 차원
TACTICAL_DROPOUT = 0.3  # 전술 드롭아웃 30%


class SpatialAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.goal_attn = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
        self.zone_attn = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1))
        self.pos_attn = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
        self.fusion = nn.Linear(3, 1)
    def forward(self, x):
        return self.fusion(torch.cat([self.pos_attn(x[..., 0:2]), 
                                       self.goal_attn(x[..., 11:14]), 
                                       self.zone_attn(x[..., 16:19])], dim=-1))


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 100, hidden_dim) * 0.02)
        self.temporal_attn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), 
                                           nn.Tanh(), nn.Dropout(0.1), 
                                           nn.Linear(hidden_dim // 2, 1))
    def forward(self, lstm_out):
        return self.temporal_attn(lstm_out + self.pos_encoding[:, :lstm_out.size(1), :])


class SpatialTemporalFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.spatial_weight = nn.Parameter(torch.tensor(0.5))
        self.temporal_weight = nn.Parameter(torch.tensor(0.5))
        self.combine = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
    def forward(self, s, t):
        return (torch.sigmoid(self.spatial_weight) * s + 
                torch.sigmoid(self.temporal_weight) * t) * self.combine(torch.cat([s, t], -1))


class ImprovedMDNPredictor(nn.Module):
    """
    [V16] 물리 엔진 + 전술 직관 분리 모델
    
    🎯 핵심 구조:
    1️⃣ 물리 엔진 (The Eye): LSTM - 순수 움직임만 학습
    2️⃣ 전술 직관 (The Brain): Embeddings - N-gram + Team ID
    3️⃣ 전술 드롭아웃: 30% 확률로 전술 정보 가리기
    4️⃣ 최종 결합 (The Fusion): 물리 + 전술 → MDN
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_gaussians=2, 
                 bidirectional=True, ngram3_vocab_size=22, ngram5_vocab_size=22,
                 ngram_embed_dim=6, num_teams=12, team_embed_dim=4, tactical_dropout=0.3):
        super().__init__()
        self.num_gaussians = num_gaussians
        self.tactical_dropout_rate = tactical_dropout
        
        # 2️⃣ 전술 직관: Embeddings (The Brain)
        self.ngram3_embed = nn.Embedding(ngram3_vocab_size, ngram_embed_dim, padding_idx=0)
        self.ngram5_embed = nn.Embedding(ngram5_vocab_size, ngram_embed_dim, padding_idx=0)
        self.team_embed = nn.Embedding(num_teams + 1, team_embed_dim, padding_idx=0)
        
        # 1️⃣ 물리 엔진: LSTM (The Eye) - 순수 움직임만!
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.spatial_attn = SpatialAttention(lstm_output_dim)
        self.temporal_attn = TemporalAttention(lstm_output_dim)
        self.fusion = SpatialTemporalFusion(lstm_output_dim)
        
        # 4️⃣ 최종 결합: 물리 + 전술
        final_dim = lstm_output_dim + ngram_embed_dim*2 + team_embed_dim
        
        # MDN Heads
        self.pi_head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(final_dim // 2, num_gaussians)
        )
        self.mu_head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(final_dim // 2, num_gaussians * 2)
        )
        self.sigma_head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(final_dim // 2, num_gaussians * 2)
        )
        
        nn.init.constant_(self.mu_head[-1].bias, 0.5)
        nn.init.xavier_uniform_(self.mu_head[-1].weight, gain=0.1)

    def forward(self, x, ngram3_idx, ngram5_idx, team_ids, mask=None, lengths=None):
        batch_size, seq_len = x.size(0), x.size(1)
        
        # 3️⃣ 전술 드롭아웃 (Tactical Dropout) - 30% 확률로 가리기
        if self.training:
            # N-gram 드롭아웃
            mask_3 = (torch.rand(batch_size, seq_len, device=x.device) < self.tactical_dropout_rate)
            mask_5 = (torch.rand(batch_size, seq_len, device=x.device) < self.tactical_dropout_rate)
            ngram3_idx = torch.where(mask_3, torch.zeros_like(ngram3_idx), ngram3_idx)
            ngram5_idx = torch.where(mask_5, torch.zeros_like(ngram5_idx), ngram5_idx)
            
            # Team ID 드롭아웃
            team_mask = (torch.rand(batch_size, device=x.device) < self.tactical_dropout_rate)
            team_ids = torch.where(team_mask, torch.zeros_like(team_ids), team_ids)
        
        # 1️⃣ 물리 엔진 (The Eye) - LSTM으로 순수 움직임만 분석
        if lengths is not None:
            packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=seq_len)
        else:
            lstm_out, _ = self.lstm(x)
        
        # Attention으로 중요한 시점 가중합
        fused_attn = self.fusion(self.spatial_attn(x), self.temporal_attn(lstm_out))
        if mask is not None:
            fused_attn = fused_attn.masked_fill(mask.unsqueeze(-1), float('-inf'))
        final_attn = torch.softmax(fused_attn, dim=1)
        
        # 물리 엔진의 결론 (context)
        physics_context = torch.sum(lstm_out * final_attn, dim=1)  # (batch, lstm_output_dim)
        
        # 2️⃣ 전술 직관 (The Brain) - Embeddings
        ng3_emb = self.ngram3_embed(ngram3_idx)  # (batch, seq_len, embed_dim)
        ng5_emb = self.ngram5_embed(ngram5_idx)
        team_emb = self.team_embed(team_ids)  # (batch, team_embed_dim)
        
        # N-gram도 attention으로 가중합 (중요한 패턴만 추출)
        # 🔧 FIX: final_attn은 이미 (batch, seq, 1)이므로 unsqueeze 불필요
        ng3_context = torch.sum(ng3_emb * final_attn, dim=1)  # (batch, embed_dim)
        ng5_context = torch.sum(ng5_emb * final_attn, dim=1)
        
        # 4️⃣ 최종 결합 (The Fusion): 물리 + 전술
        # "물리적으로는 (50, 30)인데, 역습이고 서울FC니까 (55, 30)으로 조정"
        final_features = torch.cat([physics_context, ng3_context, ng5_context, team_emb], dim=-1)
        
        # MDN 예측
        pi = torch.softmax(self.pi_head(final_features), dim=1)
        mu = torch.sigmoid(self.mu_head(final_features)).view(batch_size, self.num_gaussians, 2)
        sigma_raw = self.sigma_head(final_features).view(batch_size, self.num_gaussians, 2)
        sigma = torch.sigmoid(sigma_raw) * (MAX_SIGMA - MIN_SIGMA) + MIN_SIGMA
        
        return pi, mu, sigma


def hybrid_mdn_loss(pi, mu, sigma, target, mse_weight=HYBRID_LOSS_WEIGHT):
    """Hybrid Loss: NLL + MSE"""
    batch_size = target.size(0)
    target_expanded = target.unsqueeze(1).expand_as(mu)
    
    # NLL Loss
    diff = target_expanded - mu
    log_prob_components = (
        -0.5 * math.log(2 * math.pi) - torch.log(sigma) - 0.5 * (diff / sigma) ** 2
    )
    log_prob = log_prob_components.sum(dim=2)
    weighted_log_prob = log_prob + torch.log(pi + 1e-8)
    nll_loss = -torch.logsumexp(weighted_log_prob, dim=1).mean()
    
    # MSE Loss
    pred_mean = (pi.unsqueeze(-1) * mu).sum(dim=1)
    mse_loss = nn.functional.mse_loss(pred_mean, target)
    
    total_loss = nll_loss + mse_weight * mse_loss
    
    return total_loss, nll_loss, mse_loss


def mdn_predict_improved(pi, mu, sigma, strategy='mean'):
    if strategy == 'mode':
        max_idx = torch.argmax(pi, dim=1)
        pred = mu[torch.arange(len(mu)), max_idx]
    elif strategy == 'mean':
        pred = (pi.unsqueeze(-1) * mu).sum(dim=1)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    return torch.clamp(pred, 0.0, 1.0)


# 모델 생성
NGRAM3_VOCAB_SIZE = len(TOP_3GRAMS) + 2
NGRAM5_VOCAB_SIZE = len(TOP_5GRAMS) + 2

model = ImprovedMDNPredictor(
    INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS, 
    BIDIRECTIONAL,
    ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
    ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
    ngram_embed_dim=NGRAM_EMBED_DIM,
    num_teams=NUM_TEAMS_ACTUAL,  # 🔧 실제 팀 개수 사용
    team_embed_dim=TEAM_EMBED_DIM,
    tactical_dropout=TACTICAL_DROPOUT
).to(DEVICE)

print("="*70)
print("✅ [V16] 물리 엔진 + 전술 직관 분리 모델")
print("="*70)
print(f"🎯 핵심 구조:")
print(f"  1️⃣ 물리 엔진 (LSTM): 순수 움직임만 학습")
print(f"  2️⃣ 전술 직관 (Embeddings):")
print(f"     - 3-gram: vocab {NGRAM3_VOCAB_SIZE}, dim {NGRAM_EMBED_DIM}")
print(f"     - 5-gram: vocab {NGRAM5_VOCAB_SIZE}, dim {NGRAM_EMBED_DIM}")
print(f"     - Team ID: vocab {NUM_TEAMS_ACTUAL+1}, dim {TEAM_EMBED_DIM}")
print(f"  3️⃣ 전술 드롭아웃: {TACTICAL_DROPOUT*100:.0f}% (과적합 방지)")
print(f"  4️⃣ Head Injection: LSTM 후 MDN 직전 결합")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("="*70)

✅ [V16] 물리 엔진 + 전술 직관 분리 모델
🎯 핵심 구조:
  1️⃣ 물리 엔진 (LSTM): 순수 움직임만 학습
  2️⃣ 전술 직관 (Embeddings):
     - 3-gram: vocab 22, dim 6
     - 5-gram: vocab 22, dim 6
     - Team ID: vocab 13, dim 4
  3️⃣ 전술 드롭아웃: 30% (과적합 방지)
  4️⃣ Head Injection: LSTM 후 MDN 직전 결합
  Parameters: 4,387,929


In [8]:
# ==================== 5-Fold 학습 (모델 저장 전용) ====================
# 목적:
# - fold별 best 모델을 v17_lstm_fold{fold}.pth 로 저장
# - 추후 재실행/재사용을 위해 split 정보도 저장
#
# ✅ 중요: Validation에서 MDN 출력(mu[:,0,:])을 그대로 쓰면 성능 선별이 왜곡될 수 있음
#          mdn_predict_improved로 (pi 가중) 예측을 사용

from sklearn.model_selection import KFold
import pickle

PRED_STRATEGY = 'mean'
USE_Y_AUGMENTATION = True

# 🆕 Early Stopping 설정
EARLY_STOPPING_PATIENCE = 10  # 10 epoch 동안 개선 없으면 조기 종료

print("=" * 60)
print("🎯 5-Fold Training (모델 저장) - Leakage 차단!")
print(f"   Early Stopping Patience: {EARLY_STOPPING_PATIENCE} epochs")
print("=" * 60)

# Episode ID 목록 (중복 제거)
episode_ids_array = train_df[ID_COL].unique()
print(f"\n총 에피소드 수: {len(episode_ids_array)}")

# KFold 생성
N_SPLITS = 5
KFOLD_RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=KFOLD_RANDOM_STATE)

# split 재현을 위해 저장
fold_splits = []  # list of dicts: {fold, train_idx, val_idx}

# 각 fold별 성능 기록
fold_performances = []

for fold, (train_idx, val_idx) in enumerate(kf.split(episode_ids_array)):
    print(f"\n{'='*60}")
    print(f"📁 Fold {fold+1}/{N_SPLITS}")
    print(f"{'='*60}")

    fold_splits.append({
        'fold': int(fold),
        'train_idx': train_idx.tolist(),
        'val_idx': val_idx.tolist(),
    })

    # 1. Episode ID 기준으로 train/val split
    train_episodes = episode_ids_array[train_idx]
    val_episodes = episode_ids_array[val_idx]

    train_subset_df = train_df[train_df[ID_COL].isin(train_episodes)]
    val_subset_df = train_df[train_df[ID_COL].isin(val_episodes)]

    print(f"Train Episodes: {len(train_episodes)} ({len(train_subset_df)} rows)")
    print(f"Val Episodes: {len(val_episodes)} ({len(val_subset_df)} rows)")

    # 2. Dataset 생성
    # Train: Y축 증강 적용
    train_dataset = SoccerDataset(train_subset_df, mode='train', augment_y=USE_Y_AUGMENTATION)
    # Val: 증강 없이 (검증용)
    val_dataset = SoccerDataset(val_subset_df, mode='val', augment_y=False)

    print(f"Train Dataset: {len(train_dataset)} (증강 포함)")
    print(f"Val Dataset: {len(val_dataset)} (검증용)")

    # 3. DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)

    # 4. 모델 초기화
    model = ImprovedMDNPredictor(
        INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS,
        BIDIRECTIONAL,
        ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
        ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
        ngram_embed_dim=NGRAM_EMBED_DIM,
        num_teams=NUM_TEAMS_ACTUAL,
        team_embed_dim=TEAM_EMBED_DIM,
        tactical_dropout=TACTICAL_DROPOUT
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)

    # 5. 학습 루프
    best_dist = float('inf')
    history = {'train_loss': [], 'val_dist': []}
    
    # 🆕 Early Stopping 변수
    patience_counter = 0
    best_epoch = 0

    for epoch in range(EPOCHS_BASE):
        # Warmup
        if epoch < WARMUP_EPOCHS:
            lr = LR_BASE * (epoch + 1) / WARMUP_EPOCHS
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

        # Training
        model.train()
        train_losses = []
        train_nlls = []
        train_mses = []
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_BASE} [Train]", leave=False):
            seqs, ng3, ng5, team_ids, targets, mask, lens, _ = batch
            seqs = seqs.to(DEVICE)
            ng3 = ng3.to(DEVICE)
            ng5 = ng5.to(DEVICE)
            team_ids = team_ids.to(DEVICE)
            targets = targets.to(DEVICE)
            mask = mask.to(DEVICE)
            lens = lens.to(DEVICE)

            optimizer.zero_grad()
            pi, mu, sigma = model(seqs, ng3, ng5, team_ids, mask, lens)
            loss, nll, mse = hybrid_mdn_loss(pi, mu, sigma, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            train_losses.append(loss.item())
            train_nlls.append(nll.item())
            train_mses.append(mse.item())

        # Validation
        model.eval()
        val_dists = []
        with torch.no_grad():
            for batch in val_loader:
                seqs, ng3, ng5, team_ids, targets, mask, lens, _ = batch
                seqs = seqs.to(DEVICE)
                ng3 = ng3.to(DEVICE)
                ng5 = ng5.to(DEVICE)
                team_ids = team_ids.to(DEVICE)
                targets = targets.to(DEVICE)
                mask = mask.to(DEVICE)
                lens = lens.to(DEVICE)

                pi, mu, sigma = model(seqs, ng3, ng5, team_ids, mask, lens)
                pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)

                # 실제 미터 단위로 변환하여 거리 계산
                pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                targets_real = targets.cpu().numpy() * np.array([105.0, 68.0])
                dists = np.sqrt(np.sum((pred_real - targets_real) ** 2, axis=1))
                val_dists.extend(dists)

        avg_train_loss = np.mean(train_losses)
        avg_train_nll = np.mean(train_nlls)
        avg_train_mse = np.mean(train_mses)
        avg_val_dist = np.mean(val_dists)
        history['train_loss'].append(avg_train_loss)
        history['val_dist'].append(avg_val_dist)

        # 🆕 Early Stopping 체크
        is_best = avg_val_dist < best_dist
        if is_best:
            best_dist = avg_val_dist
            best_epoch = epoch + 1
            patience_counter = 0  # 개선되면 카운터 리셋
            torch.save(model.state_dict(), f'v17_lstm_fold{fold}.pth')
        else:
            patience_counter += 1

        # 현재 Learning Rate
        current_lr = optimizer.param_groups[0]['lr']

        # 매 epoch 로그 출력
        best_marker = "⭐" if is_best else ""
        early_stop_info = f"| ES: {patience_counter}/{EARLY_STOPPING_PATIENCE}" if patience_counter > 0 else ""
        print(
            f"[Epoch {epoch+1:2d}/{EPOCHS_BASE}] Loss: {avg_train_loss:7.4f} "
            f"(NLL: {avg_train_nll:7.4f}, MSE: {avg_train_mse:.4f}) | "
            f"Val: {avg_val_dist:7.4f}m | LR: {current_lr:.6f} {best_marker} {early_stop_info}"
        )

        # 🆕 Early Stopping 조건
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⏸️ Early stopping triggered at epoch {epoch+1}")
            print(f"   Best epoch: {best_epoch} (Val: {best_dist:.4f}m)")
            break

        scheduler.step()

    print(f"\n✅ Fold {fold+1} 최고 성능: {best_dist:.4f}m (Epoch {best_epoch})")

    fold_performances.append({
        'fold': fold + 1,
        'best_dist': float(best_dist),
        'best_epoch': best_epoch,
        'total_epochs': epoch + 1,
        'history': history,
    })

# split/로그 저장 (재사용 목적)
print(f"\n💾 KFold split 저장 중...")
with open('v17_kfold_splits.pkl', 'wb') as f:
    pickle.dump({
        'episode_ids': episode_ids_array.tolist(),
        'splits': fold_splits,
        'n_splits': int(N_SPLITS),
        'shuffle': True,
        'random_state': int(KFOLD_RANDOM_STATE),
    }, f)
print("✅ v17_kfold_splits.pkl 저장 완료")

print(f"\n💾 Fold 성능 로그 저장 중...")
with open('v17_fold_performances.pkl', 'wb') as f:
    pickle.dump(fold_performances, f)
print("✅ v17_fold_performances.pkl 저장 완료")

# 성능 요약
print(f"\n{'='*60}")
print("📊 5-Fold 학습 성능 요약")
print(f"{'='*60}")
for perf in fold_performances:
    print(f"Fold {perf['fold']}: Best Val Dist = {perf['best_dist']:.4f}m (Epoch {perf['best_epoch']}/{perf['total_epochs']})")
print(f"평균: {np.mean([p['best_dist'] for p in fold_performances]):.4f}m")
print(f"총 학습 epoch (평균): {np.mean([p['total_epochs'] for p in fold_performances]):.1f}")


🎯 5-Fold Training (모델 저장) - Leakage 차단!
   Early Stopping Patience: 7 epochs

총 에피소드 수: 15435

📁 Fold 1/5
Train Episodes: 12348 (284470 rows)
Val Episodes: 3087 (72251 rows)


Dataset (val): 100%|██████████| 3087/3087 [00:03<00:00, 846.67it/s]


Train Dataset: 24684 (증강 포함)
Val Dataset: 3087 (검증용)


[Epoch  1/50] Loss:  0.0741 (NLL:  0.0468, MSE: 0.0909) | Val: 31.9790m | LR: 0.000333 ⭐ 


[Epoch  2/50] Loss: -0.5602 (NLL: -0.5826, MSE: 0.0747) | Val: 18.0473m | LR: 0.000667 ⭐ 


...

[Epoch 23/50] Loss: -2.2652 (NLL: -2.2718, MSE: 0.0220) | Val: 13.7489m | LR: 0.000596  | ES: 3/7


[Epoch 24/50] Loss: -2.3165 (NLL: -2.3229, MSE: 0.0216) | Val: 13.9712m | LR: 0.000565  | ES: 4/7


[Epoch 25/50] Loss: -2.3462 (NLL: -2.3526, MSE: 0.0212) | Val: 13.8662m | LR: 0.000533  | ES: 5/7


[Epoch 26/50] Loss: -2.3940 (NLL: -2.4002, MSE: 0.0207) | Val: 13.8610m | LR: 0.000502  | ES: 6/7


[Epoch 27/50] Loss: -2.4423 (NLL: -2.4484, MSE: 0.0204) | Val: 13.8100m | LR: 0.000470  | ES: 7/7

⏸️ Early stopping triggered at epoch 27
   Best epoch: 20 (Val: 13.7070m)

✅ Fold 5 최고 성능: 13.7070m (Epoch 20)

💾 KFold split 저장 중...
✅ v17_kfold_splits.pkl 저장 완료

💾 Fold 성능 로그 저장 중...
✅ v17_fold_performances.pkl 저장 완료

📊 5-Fold 학습 성능 요약
Fold 1: Best Val Dist = 13.4995m (Epoch 20/27)
Fold 2: Best Val Dist = 13.4949m (Epoch 22/29)
Fold 3: Best Val Dist = 13.7902m (Epoch 20/27)
Fold 4: Best Val Dist = 14.3098m (Epoch 28/35)
Fold 5: Best Val Dist = 13.7070m (Epoch 20/27)
평균: 13.7603m
총 학습 epoch (평균): 29.0


In [9]:
# ==================== 5-Fold 앙상블 추론 ====================
print("=" * 70)
print("🎯 5-Fold 앙상블 추론")
print("=" * 70)

# 테스트 데이터 로더 생성
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)

# 5개 모델의 예측값을 저장할 리스트
all_fold_predictions = []
episode_ids_order = []  # episode_id 순서 저장

N_SPLITS = 5
for fold in range(N_SPLITS):
    print(f"\n📁 Loading Fold {fold} model...")
    
    # 모델 초기화
    model = ImprovedMDNPredictor(
        INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS,
        BIDIRECTIONAL,
        ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
        ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
        ngram_embed_dim=NGRAM_EMBED_DIM,
        num_teams=NUM_TEAMS_ACTUAL,
        team_embed_dim=TEAM_EMBED_DIM,
        tactical_dropout=TACTICAL_DROPOUT
    ).to(DEVICE)
    
    # 모델 가중치 로드
    model_path = f'v17_lstm_fold{fold}.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f"✅ Loaded: {model_path}")
    else:
        print(f"⚠️ Warning: {model_path} not found!")
        continue
    
    # 추론 모드
    model.eval()
    fold_predictions = []
    fold_episode_ids = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold} Inference"):
            seqs, ng3, ng5, team_ids, mask, lens, episode_ids = batch
            seqs = seqs.to(DEVICE)
            ng3 = ng3.to(DEVICE)
            ng5 = ng5.to(DEVICE)
            team_ids = team_ids.to(DEVICE)
            mask = mask.to(DEVICE)
            lens = lens.to(DEVICE)
            
            # MDN 예측
            pi, mu, sigma = model(seqs, ng3, ng5, team_ids, mask, lens)
            pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)
            
            # 실제 좌표로 변환 (normalized → meters)
            pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
            fold_predictions.append(pred_real)
            
            # Fold 0에서만 episode_id 순서 저장
            if fold == 0:
                fold_episode_ids.extend(episode_ids)
    
    # Fold 예측값 병합
    fold_predictions = np.vstack(fold_predictions)
    all_fold_predictions.append(fold_predictions)
    
    # Episode ID는 한 번만 저장
    if fold == 0:
        episode_ids_order = fold_episode_ids
    
    print(f"✅ Fold {fold} 예측 완료: {fold_predictions.shape}")

# 5-Fold 앙상블 (평균)
print(f"\n{'='*70}")
print("🎯 5-Fold 앙상블 중...")
print(f"{'='*70}")

all_fold_predictions = np.array(all_fold_predictions)  # (5, N, 2)
ensemble_predictions = np.mean(all_fold_predictions, axis=0)  # (N, 2)

print(f"✅ 앙상블 완료: {ensemble_predictions.shape}")
print(f"   각 Fold 예측: {all_fold_predictions.shape}")
print(f"   최종 예측 (평균): {ensemble_predictions.shape}")

# 제출 파일 생성
submit_df = pd.DataFrame({
    'game_episode': episode_ids_order,
    'end_x': ensemble_predictions[:, 0],
    'end_y': ensemble_predictions[:, 1]
})

# 저장
submit_filename = 'v17_submit_5fold_ensemble.csv'
submit_df.to_csv(submit_filename, index=False)

print(f"\n{'='*70}")
print(f"✅ 제출 파일 생성 완료!")
print(f"{'='*70}")
print(f"   파일명: {submit_filename}")
print(f"   행 개수: {len(submit_df)}")
print(f"\n📊 예측값 통계:")
print(f"   end_x: min={submit_df['end_x'].min():.2f}, max={submit_df['end_x'].max():.2f}, mean={submit_df['end_x'].mean():.2f}")
print(f"   end_y: min={submit_df['end_y'].min():.2f}, max={submit_df['end_y'].max():.2f}, mean={submit_df['end_y'].mean():.2f}")
print(f"\n🎉 제출 준비 완료!")

🎯 5-Fold 앙상블 추론

📁 Loading Fold 0 model...
✅ Loaded: v17_lstm_fold0.pth


Fold 0 Inference: 100%|██████████| 38/38 [00:00<00:00, 73.93it/s]


✅ Fold 0 예측 완료: (2414, 2)

📁 Loading Fold 1 model...
✅ Loaded: v17_lstm_fold1.pth


Fold 1 Inference: 100%|██████████| 38/38 [00:00<00:00, 76.77it/s]


✅ Fold 1 예측 완료: (2414, 2)

📁 Loading Fold 2 model...
✅ Loaded: v17_lstm_fold2.pth


...

✅ Fold 2 예측 완료: (2414, 2)

📁 Loading Fold 3 model...
✅ Loaded: v17_lstm_fold3.pth


Fold 3 Inference: 100%|██████████| 38/38 [00:00<00:00, 75.85it/s]


✅ Fold 3 예측 완료: (2414, 2)

📁 Loading Fold 4 model...
✅ Loaded: v17_lstm_fold4.pth


Fold 4 Inference: 100%|██████████| 38/38 [00:00<00:00, 79.50it/s]

✅ Fold 4 예측 완료: (2414, 2)

🎯 5-Fold 앙상블 중...
✅ 앙상블 완료: (2414, 2)
   각 Fold 예측: (5, 2414, 2)
   최종 예측 (평균): (2414, 2)

✅ 제출 파일 생성 완료!
   파일명: v17_submit_5fold_ensemble.csv
   행 개수: 2414

📊 예측값 통계:
   end_x: min=4.79, max=101.92, mean=67.34
   end_y: min=0.06, max=67.89, mean=33.40

🎉 제출 준비 완료!


In [17]:
# ==================== 🚀 TTA: Y축 반전 증강 추론 ====================
print("=" * 70)
print("🚀 TTA (Test Time Augmentation) - Y축 반전 증강")
print("=" * 70)
print("✅ 학습 때 Y축 반전 증강을 사용했으므로, 추론 때도 동일하게 적용!")
print("🎯 방법: 원본 + Y축 반전 → 두 예측을 평균")
print("=" * 70)

# 1️⃣ Y축 반전 테스트 데이터 생성
print("\n📊 Y축 반전 테스트 데이터 생성 중...")
test_df_flipped = test_df.copy()
test_df_flipped['start_y'] = 68.0 - test_df_flipped['start_y']
test_df_flipped['end_y'] = 68.0 - test_df_flipped['end_y']

# 반전된 dataset 생성
from tqdm.auto import tqdm
test_dataset_flipped = SoccerDataset(test_df_flipped, mode='test', augment_y=False)
print(f"✅ Y축 반전 Test Dataset: {len(test_dataset_flipped)} episodes")

# DataLoader 생성
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)
test_loader_flipped = DataLoader(test_dataset_flipped, batch_size=BATCH_SIZE, shuffle=False,
                                collate_fn=collate_fn, num_workers=0)

# 2️⃣ 5-Fold TTA 추론
all_fold_predictions_tta = []
episode_ids_order = []

N_SPLITS = 5
for fold in range(N_SPLITS):
    print(f"\n{'='*70}")
    print(f"📁 Fold {fold} - TTA 추론")
    print(f"{'='*70}")
    
    # 모델 초기화 및 로드
    model = ImprovedMDNPredictor(
        INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS,
        BIDIRECTIONAL,
        ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
        ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
        ngram_embed_dim=NGRAM_EMBED_DIM,
        num_teams=NUM_TEAMS_ACTUAL,
        team_embed_dim=TEAM_EMBED_DIM,
        tactical_dropout=TACTICAL_DROPOUT
    ).to(DEVICE)
    
    model_path = f'v17_lstm_fold{fold}.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f"✅ Loaded: {model_path}")
    else:
        print(f"⚠️ Warning: {model_path} not found!")
        continue
    
    model.eval()
    
    # 📍 원본 예측
    print("   1️⃣ 원본 데이터 추론...")
    pred_original_list = []
    fold_episode_ids = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Original", leave=False):
            seqs, ng3, ng5, team_ids, mask, lens, episode_ids = batch
            seqs = seqs.to(DEVICE)
            ng3 = ng3.to(DEVICE)
            ng5 = ng5.to(DEVICE)
            team_ids = team_ids.to(DEVICE)
            mask = mask.to(DEVICE)
            lens = lens.to(DEVICE)
            
            pi, mu, sigma = model(seqs, ng3, ng5, team_ids, mask, lens)
            pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)
            pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
            pred_original_list.append(pred_real)
            
            if fold == 0:
                fold_episode_ids.extend(episode_ids)
    
    pred_original = np.vstack(pred_original_list)
    
    # 📍 Y축 반전 예측
    print("   2️⃣ Y축 반전 데이터 추론...")
    pred_flipped_list = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader_flipped, desc=f"Y-Flipped", leave=False):
            seqs, ng3, ng5, team_ids, mask, lens, episode_ids = batch
            seqs = seqs.to(DEVICE)
            ng3 = ng3.to(DEVICE)
            ng5 = ng5.to(DEVICE)
            team_ids = team_ids.to(DEVICE)
            mask = mask.to(DEVICE)
            lens = lens.to(DEVICE)
            
            pi, mu, sigma = model(seqs, ng3, ng5, team_ids, mask, lens)
            pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)
            pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
            pred_flipped_list.append(pred_real)
    
    pred_flipped_raw = np.vstack(pred_flipped_list)
    
    # 3️⃣ Y 좌표 복원 (반전된 예측을 다시 뒤집기)
    pred_flipped = pred_flipped_raw.copy()
    pred_flipped[:, 1] = 68.0 - pred_flipped_raw[:, 1]
    
    # 4️⃣ TTA: 두 예측의 평균
    pred_tta = (pred_original + pred_flipped) / 2.0
    
    all_fold_predictions_tta.append(pred_tta)
    
    if fold == 0:
        episode_ids_order = fold_episode_ids
    
    print(f"   ✅ TTA 완료: {pred_tta.shape}")
    print(f"      원본 예측 범위: X[{pred_original[:, 0].min():.2f}, {pred_original[:, 0].max():.2f}], Y[{pred_original[:, 1].min():.2f}, {pred_original[:, 1].max():.2f}]")
    print(f"      반전 예측 범위: X[{pred_flipped[:, 0].min():.2f}, {pred_flipped[:, 0].max():.2f}], Y[{pred_flipped[:, 1].min():.2f}, {pred_flipped[:, 1].max():.2f}]")
    print(f"      TTA 예측 범위: X[{pred_tta[:, 0].min():.2f}, {pred_tta[:, 0].max():.2f}], Y[{pred_tta[:, 1].min():.2f}, {pred_tta[:, 1].max():.2f}]")

# 5️⃣ 5-Fold 앙상블
print(f"\n{'='*70}")
print("🎯 5-Fold TTA 앙상블 중...")
print(f"{'='*70}")

all_fold_predictions_tta = np.array(all_fold_predictions_tta)  # (5, N, 2)
ensemble_predictions_tta = np.mean(all_fold_predictions_tta, axis=0)  # (N, 2)

print(f"✅ TTA 앙상블 완료: {ensemble_predictions_tta.shape}")
print(f"   각 Fold TTA 예측: {all_fold_predictions_tta.shape}")
print(f"   최종 TTA 예측 (평균): {ensemble_predictions_tta.shape}")

# 6️⃣ 제출 파일 생성
submit_df_tta = pd.DataFrame({
    'game_episode': episode_ids_order,
    'end_x': ensemble_predictions_tta[:, 0],
    'end_y': ensemble_predictions_tta[:, 1]
})

submit_filename_tta = 'v17_submit_5fold_TTA.csv'
submit_df_tta.to_csv(submit_filename_tta, index=False)

print(f"\n{'='*70}")
print(f"✅ TTA 제출 파일 생성 완료!")
print(f"{'='*70}")
print(f"   파일명: {submit_filename_tta}")
print(f"   행 개수: {len(submit_df_tta)}")
print(f"\n📊 TTA 예측값 통계:")
print(f"   end_x: min={submit_df_tta['end_x'].min():.2f}, max={submit_df_tta['end_x'].max():.2f}, mean={submit_df_tta['end_x'].mean():.2f}")
print(f"   end_y: min={submit_df_tta['end_y'].min():.2f}, max={submit_df_tta['end_y'].max():.2f}, mean={submit_df_tta['end_y'].mean():.2f}")

# 7️⃣ 기존 예측 vs TTA 예측 비교
if 'submit_df' in globals():
    print(f"\n{'='*70}")
    print("📊 기존 예측 vs TTA 예측 비교")
    print(f"{'='*70}")
    
    # 예측값 차이 계산
    diff_x = np.abs(submit_df_tta['end_x'].values - submit_df['end_x'].values)
    diff_y = np.abs(submit_df_tta['end_y'].values - submit_df['end_y'].values)
    diff_total = np.sqrt(diff_x**2 + diff_y**2)
    
    print(f"📍 평균 거리 차이: {diff_total.mean():.4f}m")
    print(f"   X 좌표 차이: {diff_x.mean():.4f}m (max: {diff_x.max():.4f}m)")
    print(f"   Y 좌표 차이: {diff_y.mean():.4f}m (max: {diff_y.max():.4f}m)")
    print(f"\n💡 TTA 효과: Y축 대칭성을 활용해 편향 제거 + 불확실성 감소")

print(f"\n🎉 TTA 추론 완료! 재학습 없이 공짜로 성능 개선! 🚀")

🚀 TTA (Test Time Augmentation) - Y축 반전 증강
✅ 학습 때 Y축 반전 증강을 사용했으므로, 추론 때도 동일하게 적용!
🎯 방법: 원본 + Y축 반전 → 두 예측을 평균

📊 Y축 반전 테스트 데이터 생성 중...


Dataset (test): 100%|██████████| 2414/2414 [00:02<00:00, 941.50it/s] 


✅ Y축 반전 Test Dataset: 2414 episodes

📁 Fold 0 - TTA 추론
✅ Loaded: v17_lstm_fold0.pth
   1️⃣ 원본 데이터 추론...


   2️⃣ Y축 반전 데이터 추론...


   ✅ TTA 완료: (2414, 2)
      원본 예측 범위: X[2.30, 102.46], Y[0.03, 67.87]
      반전 예측 범위: X[2.98, 103.14], Y[0.07, 67.97]
      TTA 예측 범위: X[3.81, 102.63], Y[0.05, 67.91]

📁 Fold 1 - TTA 추론
✅ Loaded: v17_lstm_fold1.pth
   1️⃣ 원본 데이터 추론...


...

   ✅ TTA 완료: (2414, 2)
      원본 예측 범위: X[6.29, 101.59], Y[0.07, 67.97]
      반전 예측 범위: X[6.16, 101.57], Y[0.04, 67.94]
      TTA 예측 범위: X[6.23, 101.55], Y[0.06, 67.95]

📁 Fold 3 - TTA 추론
✅ Loaded: v17_lstm_fold3.pth
   1️⃣ 원본 데이터 추론...


   2️⃣ Y축 반전 데이터 추론...


   ✅ TTA 완료: (2414, 2)
      원본 예측 범위: X[3.14, 102.27], Y[0.03, 67.90]
      반전 예측 범위: X[3.78, 102.24], Y[0.09, 67.98]
      TTA 예측 범위: X[3.46, 102.26], Y[0.07, 67.94]

📁 Fold 4 - TTA 추론
✅ Loaded: v17_lstm_fold4.pth
   1️⃣ 원본 데이터 추론...


   2️⃣ Y축 반전 데이터 추론...


   ✅ TTA 완료: (2414, 2)
      원본 예측 범위: X[5.12, 101.86], Y[0.08, 67.95]
      반전 예측 범위: X[4.73, 102.10], Y[0.06, 67.92]
      TTA 예측 범위: X[4.92, 101.98], Y[0.08, 67.93]

🎯 5-Fold TTA 앙상블 중...
✅ TTA 앙상블 완료: (2414, 2)
   각 Fold TTA 예측: (5, 2414, 2)
   최종 TTA 예측 (평균): (2414, 2)

✅ TTA 제출 파일 생성 완료!
   파일명: v17_submit_5fold_TTA.csv
   행 개수: 2414

📊 TTA 예측값 통계:
   end_x: min=5.34, max=101.97, mean=67.35
   end_y: min=0.07, max=67.89, mean=33.54

📊 기존 예측 vs TTA 예측 비교
📍 평균 거리 차이: 0.6813m
   X 좌표 차이: 0.3287m (max: 3.3041m)
   Y 좌표 차이: 0.5037m (max: 4.4756m)

💡 TTA 효과: Y축 대칭성을 활용해 편향 제거 + 불확실성 감소

🎉 TTA 추론 완료! 재학습 없이 공짜로 성능 개선! 🚀
